In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import os
!pip install dagshub mlflow -q
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
dagshub.init(repo_owner='lkhiz23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

import mlflow
import pandas as pd
import numpy as np

print("Connected")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

Accessing as lkhiz23

Initialized MLflow to track repo "lkhiz23/IEEE-CIS-Fraud-Detection"

Repository lkhiz23/IEEE-CIS-Fraud-Detection initialized!

Connected


## Load best model from registry

In [3]:
model_uri = "models:/XGBoost_Fraud_Pipeline/1"
pipeline  = mlflow.sklearn.load_model(model_uri)

print("Model loaded:", type(pipeline))
print("Steps:", [name for name, _ in pipeline.steps])

Model loaded: <class 'sklearn.pipeline.Pipeline'>
Steps: ['drop_missing', 'feature_eng', 'label_enc', 'imputer', 'feature_selector', 'model']


In [8]:
PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
test_tx = pd.read_csv(PATH + 'test_transaction.csv')
test_id = pd.read_csv(PATH + 'test_identity.csv')

# fix column names - replace dash with underscore
test_id.columns = test_id.columns.str.replace('-', '_')

test     = test_tx.merge(test_id, on='TransactionID', how='left')
test_ids = test['TransactionID']
X_test   = test.drop(columns=['TransactionID'])

print("X_test shape:", X_test.shape)

X_test shape: (506691, 432)


## Generate Predictions

In [9]:
y_proba = pipeline.predict_proba(X_test)[:,1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud':       y_proba
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.shape)
print(submission.head())

(506691, 2)
   TransactionID   isFraud
0        3663549  0.000329
1        3663550  0.000894
2        3663551  0.005044
3        3663552  0.000956
4        3663553  0.000281
